# Configuración central del proyecto — Devoluciones en moda online

**Pregunta de negocio del proyecto:** ¿qué líneas de pedido tienen más probabilidad de ser devueltas, para ajustar guías de talla, fotos de producto y políticas de devolución?

Este notebook es el punto de partida de todo el repositorio. Aquí viven, en un solo lugar, los parámetros del proyecto: el nombre del catálogo, el tamaño de los datos sintéticos, la tasa de devolución y la semilla aleatoria. Todos los demás notebooks empiezan con `%run ./00_Setup/00_Configuracion` y heredan estas variables, de modo que cambiar un valor aquí cambia todo el proyecto sin tocar una línea de código en otro lugar.

Además de los parámetros, este notebook crea el catálogo propio del proyecto con sus tres esquemas de la arquitectura Medallion (`bronze`, `silver`, `gold`) y define dos funciones auxiliares que se reutilizan en todo el repo: `tbl()` para armar nombres de tabla y `escribir_delta()` para escribir tablas Delta siempre de la misma forma.

**En esta celda** se declaran los 7 widgets (catálogo, entorno, número de clientes, productos y líneas de pedido, tasa de devolución y semilla) para poder cambiarlos desde la barra superior del notebook sin editar código.

In [ ]:
# ============================================================
# WIDGETS DEL PROYECTO — unico lugar donde se declaran
# Todos los notebooks heredan estos valores via %run
# ============================================================

dbutils.widgets.text("catalog_name", "proyecto_final_fashion_returns", "01 · Catalog del proyecto")
dbutils.widgets.dropdown("entorno", "training", ["training", "dev", "demo"], "02 · Entorno")
dbutils.widgets.text("num_clientes", "5000", "03 · Clientes sinteticos")
dbutils.widgets.text("num_productos", "400", "04 · Productos sinteticos")
dbutils.widgets.text("num_lineas", "20000", "05 · Lineas de pedido sinteticas")
dbutils.widgets.text("tasa_devolucion_pct", "22", "06 · Tasa base de devolucion (%)")
dbutils.widgets.text("semilla_random", "2609", "07 · Semilla aleatoria")

**En esta celda** se leen los widgets y se convierten en variables Python en mayúsculas (los widgets devuelven texto, por eso los números pasan por `int()` y `float()`), se fijan las constantes del negocio y se define `tbl()`, la función que arma el nombre completo `catalogo.schema.tabla`.

**Por qué las constantes de negocio viven aquí:** la ventana de 30 días define la etiqueta (`fue_devuelto = 1` si la devolución llega dentro de 30 días de la compra) y el máximo de 35 días es el plazo tras el cual ya se conoce toda devolución. Ambas se usan en Silver, Delta Lake, Gold y ML, así que tener un único valor evita que los notebooks se desincronicen.

In [ ]:
CATALOG_NAME = dbutils.widgets.get("catalog_name")
ENTORNO = dbutils.widgets.get("entorno")
NUM_CLIENTES = int(dbutils.widgets.get("num_clientes"))
NUM_PRODUCTOS = int(dbutils.widgets.get("num_productos"))
NUM_LINEAS = int(dbutils.widgets.get("num_lineas"))
TASA_DEVOLUCION = float(dbutils.widgets.get("tasa_devolucion_pct")) / 100
SEED = int(dbutils.widgets.get("semilla_random"))

# Arquitectura Medallion — estandar fijo, no parametrizable
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"

# Reglas de negocio compartidas por todos los notebooks
VENTANA_DEVOLUCION_DIAS = 30      # una linea cuenta como devuelta si la devolucion llega en <= 30 dias
MAX_DIAS_HASTA_DEVOLUCION = 35    # plazo maximo en el que puede llegar una devolucion
FECHA_INICIO = "2025-01-01"       # primer dia del rango de ventas sinteticas


def tbl(schema, tabla):
    """Nombre completo catalog.schema.tabla — usar SIEMPRE esta funcion en vez de hardcodear."""
    return f"{CATALOG_NAME}.{schema}.{tabla}"


print("=" * 60)
print("CONFIGURACION CARGADA")
print("=" * 60)
print(f"  catalog_name       = {CATALOG_NAME}")
print(f"  entorno            = {ENTORNO}")
print(f"  num_clientes       = {NUM_CLIENTES:,}")
print(f"  num_productos      = {NUM_PRODUCTOS:,}")
print(f"  num_lineas         = {NUM_LINEAS:,}")
print(f"  tasa_devolucion    = {TASA_DEVOLUCION:.0%}")
print(f"  semilla_random     = {SEED}")
print(f"  ventana devolucion = {VENTANA_DEVOLUCION_DIAS} dias (maximo {MAX_DIAS_HASTA_DEVOLUCION})")
print(f"  schemas medallion  = {SCHEMA_BRONZE}, {SCHEMA_SILVER}, {SCHEMA_GOLD}")

**En esta celda** se crea el catálogo propio del proyecto y sus tres esquemas, cada uno con un comentario que explica su rol.

**Por qué un catálogo propio:** aísla el proyecto de las tablas del curso, es la primera evidencia de gobernanza (se ve de un vistazo en el Catalog Explorer) y es el lugar donde después se otorgan permisos. Los comandos usan `IF NOT EXISTS`, así que se pueden correr varias veces sin errores ni duplicados.

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME} COMMENT 'Proyecto final - devoluciones en moda online ({ENTORNO})'")

comentarios = {
    SCHEMA_BRONZE: "Ingesta cruda. Datos tal como llegan del sistema origen, con sus errores. Inmutable.",
    SCHEMA_SILVER: "Datos limpios, tipados y validados. Una fila por linea de pedido.",
    SCHEMA_GOLD:   "Metricas de negocio y features para el modelo. Consumido por BI y ML.",
}
for schema, comentario in comentarios.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{schema} COMMENT '{comentario}'")
    print(f"[OK] {CATALOG_NAME}.{schema:<8} -> {comentario}")

**En esta celda** se define `escribir_delta()`, la función que escribe cualquier DataFrame como tabla Delta con el mismo patrón (patrón DRY: no repetir el mismo bloque de escritura en cada notebook). Recibe el esquema destino, el nombre de la tabla, un comentario descriptivo y, opcionalmente, una columna de partición.

**Por qué `optimize_write` es opcional:** en Bronze se activa para que los archivos nazcan de tamaño razonable. En Silver se deja apagado a propósito, para que en `02_Delta_Lake` se pueda ver con datos reales el efecto de `OPTIMIZE` sobre la cantidad de archivos.

In [ ]:
def escribir_delta(df, schema, tabla, comment, partition_col=None, optimize_write=False):
    """Escribe un DataFrame como tabla Delta (overwrite) y le pone un comentario descriptivo."""
    full_name = tbl(schema, tabla)
    writer = (
        df.write.format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
    )
    if optimize_write:
        writer = writer.option("delta.autoOptimize.optimizeWrite", "true")
    if partition_col:
        writer = writer.partitionBy(partition_col)
    writer.saveAsTable(full_name)
    spark.sql(f"COMMENT ON TABLE {full_name} IS '{comment}'")
    print(f"[OK] {full_name} — {spark.table(full_name).count():,} registros")

Setup · Configuración &nbsp;·&nbsp; Siguiente: **01_Generacion_o_Ingesta**